# 12. Gamma Framework — Central POC Question

**Three-layer framework** (per consolidated spec):
- **Layer A (pool)** : `break_even_volatility` vs `realized_volatility`, `volatility_edge`
- **Layer B (position)** : `gamma_cost_realized` (decomposed diffusion + jump), `fees_position`, FGY/GSR
- **Layer C (regime)** : RV / BV / RQ / `jump_intensity` / HAR-RV forecasts

**Pool**: USDC/WETH **0.05%** (proxy for ETH/USDC 0.3% — same math, different fee tier).
**Window**: 180 days, 2025-10-08 → 2026-04-06.

**Three POC questions** :
1. Quel **FGY sur la composante diffusion seule** ?
2. Quelle **part des pertes cumulées** est attribuable aux **jumps** ?
3. Existe-t-il un **signal leading** (HAR-RV forecast) qui aurait permis d'éviter les jours dominés par les jumps ?

**Verdict logic**:
- Si `FGY_diffusion ≥ 1` ET les jumps portent la perte → stratégie = "LP avec sortie conditionnelle sur signal anti-jumps"
- Si `FGY_diffusion < 1` → pool **structurellement non-profitable** ; hedge externe ou exit permanent


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import math
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
START_DATE, END_DATE = "2025-10-08", "2026-04-06"

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


<function __main__.<lambda>()>

## §1 — Load 3 marts

In [2]:
vol = db.read_query(f"""
    SELECT * FROM py_subjacent_vol_features_daily
    WHERE asset = 'ETH' AND date BETWEEN toDate('{START_DATE}') AND toDate('{END_DATE}')
    ORDER BY date
""").to_pandas()
pool = db.read_query(f"""
    SELECT * FROM py_pool_gamma_features_daily
    WHERE lower(pool_address) = lower('{POOL}') AND date BETWEEN toDate('{START_DATE}') AND toDate('{END_DATE}')
    ORDER BY date
""").to_pandas()
posn = db.read_query(f"""
    SELECT * FROM py_position_gamma_features_daily
    WHERE lower(pool_address) = lower('{POOL}') AND date BETWEEN toDate('{START_DATE}') AND toDate('{END_DATE}')
    ORDER BY date
""").to_pandas()

for df in (vol, pool, posn):
    df["date"] = pd.to_datetime(df["date"])

print(f"vol  : {vol.shape}  cols: {list(vol.columns)}")
print(f"pool : {pool.shape}")
print(f"posn : {posn.shape}  position_id: {posn['position_id'].unique()}")


vol  : (181, 10)  cols: ['asset', 'date', 'realized_variance_daily', 'realized_quarticity_daily', 'bipower_variation_daily', 'jump_intensity_daily', 'realized_volatility_annual_garman_klass', 'cv_realized_il', 'sigma_har_forecast_1d_annual', 'sigma_har_forecast_7d_annual']
pool : (181, 16)
posn : (362, 19)  position_id: <ArrowStringArray>
['concentrated_pm10pct_daily_rebal', 'synthetic_fitted_wide']
Length: 2, dtype: str


## §2 — Layer A : volatility_edge time-series

In [3]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    subplot_titles=("break_even_volatility vs realized_volatility (annualized)",
                    "volatility_edge (σ*_pool − σ_realized)"))
days = pool["date"]
fig.add_trace(go.Scatter(x=days, y=pool["break_even_volatility_annual_7d_window"],
                          name="σ*_pool 7d", line=dict(color="#22c55e", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=days, y=pool["realized_volatility_annual_7d_window"],
                          name="σ_realized 7d", line=dict(color="#ef4444", width=2)), row=1, col=1)
fig.add_trace(go.Bar(x=days, y=pool["volatility_edge_7d"],
                      marker_color=["#22c55e" if e is not None and e > 0 else "#ef4444" for e in pool["volatility_edge_7d"]],
                      name="edge"), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="white", row=2, col=1)
fig.update_yaxes(tickformat=".0%", row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=2, col=1)
fig.update_layout(height=560, title_text="Layer A — pool carry environment (180d)")
fig.show()

mean_edge = float(pool["volatility_edge_7d"].dropna().mean())
days_pos = int((pool["volatility_edge_7d"] > 0).sum())
days_total = int(pool["volatility_edge_7d"].notna().sum())
print(f"mean edge_7d         : {mean_edge:+.2%}")
print(f"days edge > 0        : {days_pos} / {days_total}  ({days_pos/max(days_total,1):.1%})")


mean edge_7d         : +1.30%
days edge > 0        : 88 / 181  (48.6%)


## §3 — Layer C : RV / BV / Jump intensity

In [4]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    subplot_titles=("realized_variance vs bipower_variation (daily)",
                    "jump_intensity = (RV − BV) / RV",
                    "cv_realized_il = √RQ / RV   (vol-of-vol; baseline √(3/24)≈0.354)"))
fig.add_trace(go.Scatter(x=vol["date"], y=vol["realized_variance_daily"], name="RV",
                          line=dict(color="#3b82f6")), row=1, col=1)
fig.add_trace(go.Scatter(x=vol["date"], y=vol["bipower_variation_daily"], name="BV",
                          line=dict(color="#9ca3af", dash="dot")), row=1, col=1)
fig.add_trace(go.Bar(x=vol["date"], y=vol["jump_intensity_daily"], name="jump intensity",
                      marker_color="#ef4444"), row=2, col=1)
fig.add_trace(go.Scatter(x=vol["date"], y=vol["cv_realized_il"], name="cv_realized_il",
                          line=dict(color="#a855f7")), row=3, col=1)
fig.add_hline(y=math.sqrt(3.0/24.0), line_dash="dot", line_color="white", row=3, col=1,
              annotation_text="√(3/24)≈0.354 (Gaussian iid baseline)")
fig.update_layout(height=720, title_text="Layer C — subjacent vol features (180d)")
fig.show()

mean_jump_intensity = float(vol["jump_intensity_daily"].mean())
mean_cv = float(vol["cv_realized_il"].mean())
print(f"mean jump_intensity_daily : {mean_jump_intensity:.1%}")
print(f"mean cv_realized_il       : {mean_cv:.3f}   (baseline √(3/24)={math.sqrt(3/24):.3f})")


mean jump_intensity_daily : 15.6%
mean cv_realized_il       : 0.433   (baseline √(3/24)=0.354)


## §4 — POC Question 1: FGY total / diffusion-only / jump-only

In [5]:
mean_fees = float(posn["fees_position_usd_daily"].mean())
mean_gc_total = float(posn["gamma_cost_realized_usd_daily"].mean())
mean_gc_diff  = float(posn["diffusion_gamma_cost_usd_daily"].mean())
mean_gc_jump  = float(posn["jump_gamma_cost_usd_daily"].mean())

fgy_total = mean_fees / mean_gc_total if mean_gc_total > 0 else float('nan')
fgy_diff  = mean_fees / mean_gc_diff  if mean_gc_diff  > 0 else float('nan')
fgy_jump  = mean_fees / mean_gc_jump  if mean_gc_jump  > 0 else float('nan')

print("─" * 60)
print("  POC Q1 — FGY by gamma cost component")
print("─" * 60)
print(f"  mean fees / day:                ${mean_fees:>12,.0f}")
print(f"  mean gamma_cost / day (TOTAL):  ${mean_gc_total:>12,.0f}")
print(f"  mean gamma_cost / day (diff):   ${mean_gc_diff:>12,.0f}  ({mean_gc_diff/mean_gc_total:.1%})")
print(f"  mean gamma_cost / day (jump):   ${mean_gc_jump:>12,.0f}  ({mean_gc_jump/mean_gc_total:.1%})")
print()
print(f"  FGY (total)      = mean(F) / mean(GC_total):  {fgy_total:>8.3f}")
print(f"  FGY (diffusion)  = mean(F) / mean(GC_diff):   {fgy_diff:>8.3f}")
print(f"  FGY (jump)       = mean(F) / mean(GC_jump):   {fgy_jump:>8.3f}")
print()
print(f"  Interpretation:")
print(f"    FGY_diff < 1 → diffusion alone unprofitable: pool structurally below break-even.")
print(f"    FGY_diff ≥ 1 → diffusion profitable; jumps drive losses → conditional exit may save.")
print("─" * 60)


────────────────────────────────────────────────────────────
  POC Q1 — FGY by gamma cost component
────────────────────────────────────────────────────────────
  mean fees / day:                $      59,345
  mean gamma_cost / day (TOTAL):  $      87,595
  mean gamma_cost / day (diff):   $      72,970  (83.3%)
  mean gamma_cost / day (jump):   $      14,625  (16.7%)

  FGY (total)      = mean(F) / mean(GC_total):     0.677
  FGY (diffusion)  = mean(F) / mean(GC_diff):      0.813
  FGY (jump)       = mean(F) / mean(GC_jump):      4.058

  Interpretation:
    FGY_diff < 1 → diffusion alone unprofitable: pool structurally below break-even.
    FGY_diff ≥ 1 → diffusion profitable; jumps drive losses → conditional exit may save.
────────────────────────────────────────────────────────────


## §5 — POC Question 2: jump share of cumulative losses

In [6]:
# Cumulative components
cum_fees = posn["fees_position_usd_daily"].fillna(0).cumsum()
cum_gc_total = posn["gamma_cost_realized_usd_daily"].fillna(0).cumsum()
cum_gc_diff  = posn["diffusion_gamma_cost_usd_daily"].fillna(0).cumsum()
cum_gc_jump  = posn["jump_gamma_cost_usd_daily"].fillna(0).cumsum()

cum_pnl_total = cum_fees - cum_gc_total
cum_pnl_diff_only = cum_fees - cum_gc_diff   # PnL if jumps had been hedged out
cum_pnl_jump_only = -cum_gc_jump             # losses purely from jumps

total_fees_180d = float(cum_fees.iloc[-1])
total_gc_180d   = float(cum_gc_total.iloc[-1])
total_gc_diff   = float(cum_gc_diff.iloc[-1])
total_gc_jump   = float(cum_gc_jump.iloc[-1])
total_pnl_180d  = total_fees_180d - total_gc_180d
total_pnl_diff_only = total_fees_180d - total_gc_diff

print("─" * 60)
print("  POC Q2 — Cumulative attribution over 180d")
print("─" * 60)
print(f"  Σ fees                  : ${total_fees_180d:>14,.0f}")
print(f"  Σ gamma_cost_total      : ${total_gc_180d:>14,.0f}")
print(f"    of which diffusion    : ${total_gc_diff:>14,.0f}  ({total_gc_diff/total_gc_180d:.1%})")
print(f"    of which jumps        : ${total_gc_jump:>14,.0f}  ({total_gc_jump/total_gc_180d:.1%})")
print()
print(f"  total GammaPnL          : ${total_pnl_180d:>14,.0f}")
print(f"  GammaPnL (diff only)    : ${total_pnl_diff_only:>14,.0f}    ← LP profitable here?")
print(f"  Pure jump loss          : ${-total_gc_jump:>14,.0f}")
print()
if total_pnl_diff_only > 0:
    pct_jump_in_total = (total_pnl_180d - total_pnl_diff_only) / total_pnl_180d if total_pnl_180d != 0 else float('nan')
    print(f"  → Diffusion side is PROFITABLE; jumps create the loss.")
    print(f"  → Jump-attributable share of total loss: {-total_gc_jump/(-total_pnl_180d) if total_pnl_180d < 0 else 0:.1%}")
else:
    print(f"  → Diffusion side ALONE is unprofitable. Jumps just amplify.")
print("─" * 60)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
    subplot_titles=("Cumulative Σ fees vs −Σ gamma_cost (split diff / jump)",
                    "Cumulative GammaPnL: total vs diffusion-only vs HODL-equivalent"))
fig.add_trace(go.Scatter(x=posn["date"], y=cum_fees, name="Σ fees", line=dict(color="#22c55e", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=posn["date"], y=-cum_gc_diff, name="−Σ gamma_cost_diff", line=dict(color="#3b82f6")), row=1, col=1)
fig.add_trace(go.Scatter(x=posn["date"], y=-cum_gc_jump, name="−Σ gamma_cost_jump", line=dict(color="#ef4444")), row=1, col=1)

fig.add_trace(go.Scatter(x=posn["date"], y=cum_pnl_total, name="GammaPnL total", line=dict(color="#9ca3af", width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=posn["date"], y=cum_pnl_diff_only, name="GammaPnL (diff only)", line=dict(color="#3b82f6", width=2, dash="dash")), row=2, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="white", row=2, col=1)
fig.update_layout(height=560, title_text=f"Cumulative attribution (180d) — net total ${total_pnl_180d/1e6:.2f}M")
fig.show()


────────────────────────────────────────────────────────────
  POC Q2 — Cumulative attribution over 180d
────────────────────────────────────────────────────────────
  Σ fees                  : $    21,482,983
  Σ gamma_cost_total      : $    31,709,410
    of which diffusion    : $    26,415,067  (83.3%)
    of which jumps        : $     5,294,343  (16.7%)

  total GammaPnL          : $   -10,226,427
  GammaPnL (diff only)    : $    -4,932,084    ← LP profitable here?
  Pure jump loss          : $    -5,294,343

  → Diffusion side ALONE is unprofitable. Jumps just amplify.
────────────────────────────────────────────────────────────


## §6 — POC Question 3: HAR-RV leading signal — anti-jump rule

In [7]:
# Build a synthetic out-of-sample backtest:
# Rule: skip day t if HAR-RV forecast σ²_{t} (made at t-1) > σ*²_pool_t-1.
# Compare cumulative GammaPnL vs always-LP baseline.

# Align all series on a common index
common_idx = posn["date"]
fees     = posn.set_index("date")["fees_position_usd_daily"].reindex(common_idx).fillna(0)
gc_total = posn.set_index("date")["gamma_cost_realized_usd_daily"].reindex(common_idx).fillna(0)
gc_jump  = posn.set_index("date")["jump_gamma_cost_usd_daily"].reindex(common_idx).fillna(0)
gc_diff  = posn.set_index("date")["diffusion_gamma_cost_usd_daily"].reindex(common_idx).fillna(0)
sigma_star_2 = (pool.set_index("date")["break_even_volatility_annual_7d_window"] ** 2).reindex(common_idx)
sigma_har_1d = vol.set_index("date")["sigma_har_forecast_1d_annual"].reindex(common_idx)
sigma_har_2  = sigma_har_1d ** 2

# ON-rule: LP iff sigma_har_1d² ≤ break_even²
on_rule = (sigma_har_2 <= sigma_star_2).fillna(False).astype(float)
# Coverage: how many days the rule has a forecast
coverage = float((sigma_har_1d.notna() & sigma_star_2.notna()).mean())
n_on = int(on_rule.sum())
n_total = int(sigma_har_1d.notna().sum())

# Hypothetical PnL under each rule
pnl_baseline = fees - gc_total
pnl_rule     = on_rule * pnl_baseline
cum_baseline = pnl_baseline.cumsum()
cum_rule     = pnl_rule.cumsum()

# Jump-day capture rate: of the jump days (jump_intensity > median), how many does the rule skip?
jump_intensity = vol.set_index("date")["jump_intensity_daily"].reindex(common_idx)
jump_q75 = float(jump_intensity.quantile(0.75))
high_jump_days = (jump_intensity >= jump_q75)
skipped_high_jump_days = int(((on_rule == 0) & high_jump_days & sigma_har_1d.notna()).sum())
total_high_jump_days = int((high_jump_days & sigma_har_1d.notna()).sum())

print("─" * 60)
print("  POC Q3 — HAR-RV leading-signal backtest")
print("─" * 60)
print(f"  forecast coverage:          {coverage:.1%}  (warm-up = first 60d)")
print(f"  rule LP-active days:        {n_on} / {n_total}")
print(f"  baseline cum PnL (180d):    ${float(cum_baseline.iloc[-1]):>12,.0f}")
print(f"  rule cum PnL    (180d):     ${float(cum_rule.iloc[-1]):>12,.0f}")
print(f"  Δ (rule − baseline):        ${float(cum_rule.iloc[-1] - cum_baseline.iloc[-1]):>12,.0f}")
print()
print(f"  high-jump days (jump_intensity ≥ p75):  {total_high_jump_days}")
print(f"  among them, skipped by rule:            {skipped_high_jump_days} ({skipped_high_jump_days/max(total_high_jump_days,1):.0%})")
print("─" * 60)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.07,
    subplot_titles=("Cumulative GammaPnL: baseline (always-LP) vs HAR-RV rule",
                    "Daily on/off (1 = LP active, 0 = sit out)"))
fig.add_trace(go.Scatter(x=common_idx, y=cum_baseline, name="baseline always-LP", line=dict(color="#9ca3af", width=2)), row=1, col=1)
fig.add_trace(go.Scatter(x=common_idx, y=cum_rule, name="HAR-RV rule", line=dict(color="#22c55e", width=2)), row=1, col=1)
fig.add_hline(y=0, line_dash="dash", line_color="white", row=1, col=1)
fig.add_trace(go.Bar(x=common_idx, y=on_rule, name="on/off",
                      marker_color=["#22c55e" if v else "#ef4444" for v in on_rule.fillna(0).astype(int)]), row=2, col=1)
fig.update_yaxes(range=[-0.1, 1.1], row=2, col=1)
fig.update_layout(height=560, title_text="HAR-RV anti-jump rule backtest (180d)", showlegend=True)
fig.show()


────────────────────────────────────────────────────────────
  POC Q3 — HAR-RV leading-signal backtest
────────────────────────────────────────────────────────────
  forecast coverage:          100.0%  (warm-up = first 60d)
  rule LP-active days:        214 / 362
  baseline cum PnL (180d):    $ -10,226,427
  rule cum PnL    (180d):     $ -13,236,444
  Δ (rule − baseline):        $  -3,010,017

  high-jump days (jump_intensity ≥ p75):  92
  among them, skipped by rule:            36 (39%)
────────────────────────────────────────────────────────────


## §7 — Verdict

In [8]:
print("═" * 72)
print("                       FRAMEWORK VERDICT")
print("═" * 72)
print()
print("  Pool:    USDC/WETH 0.05% (proxy for ETH/USDC 0.3%)")
print("  Window:  180 days (2025-10-08 → 2026-04-06)")
print()
print("─ Layer A (pool) ──────────────────────────────────────────────────")
print(f"  mean σ*_pool_7d:        {float(pool['break_even_volatility_annual_7d_window'].mean()):>6.1%}")
print(f"  mean σ_realized_7d:     {float(pool['realized_volatility_annual_7d_window'].mean()):>6.1%}")
print(f"  mean edge:              {float(pool['volatility_edge_7d'].dropna().mean()):>+6.2%}")
print(f"  days edge > 0:          {int((pool['volatility_edge_7d'] > 0).sum())} / {int(pool['volatility_edge_7d'].notna().sum())}")
print()
print("─ Layer B (position) ──────────────────────────────────────────────")
print(f"  FGY (total):            {fgy_total:>6.3f}")
print(f"  FGY (diffusion only):   {fgy_diff:>6.3f}")
print(f"  FGY (jump only):        {fgy_jump:>6.3f}")
print(f"  Σ jump_gc / Σ gc_total: {total_gc_jump/total_gc_180d:>6.1%}")
print()
print("─ Layer C (regime) ────────────────────────────────────────────────")
print(f"  mean cv_realized_il:    {float(vol['cv_realized_il'].mean()):>6.3f}  (baseline {math.sqrt(3/24):.3f})")
print(f"  mean jump_intensity:    {float(vol['jump_intensity_daily'].mean()):>6.1%}")
print()
print("─ HAR-RV anti-jump rule ────────────────────────────────────────────")
har_lift = float(cum_rule.iloc[-1] - cum_baseline.iloc[-1])
print(f"  cumulative PnL Δ:       ${har_lift:>+12,.0f}")
print(f"  high-jump days skipped: {skipped_high_jump_days}/{total_high_jump_days}")
print()
print("═" * 72)
print("                          STRATEGIC READING")
print("═" * 72)
if fgy_diff >= 1.0:
    print("  ✓ FGY_diffusion ≥ 1 → diffusion side profitable.")
    print("    → Conditional exit on jump signal could rescue the strategy.")
    print(f"    → HAR-RV lift = ${har_lift:+,.0f}: " + ("works" if har_lift > 0 else "INSUFFICIENT"))
else:
    print("  ✗ FGY_diffusion < 1 → diffusion alone is below break-even.")
    print("    Pool is structurally unprofitable in this regime.")
    print("    Jump signal cannot rescue it. Options: hedge externally or exit.")
print("═" * 72)


════════════════════════════════════════════════════════════════════════
                       FRAMEWORK VERDICT
════════════════════════════════════════════════════════════════════════

  Pool:    USDC/WETH 0.05% (proxy for ETH/USDC 0.3%)
  Window:  180 days (2025-10-08 → 2026-04-06)

─ Layer A (pool) ──────────────────────────────────────────────────
  mean σ*_pool_7d:         63.9%
  mean σ_realized_7d:      62.6%
  mean edge:              +1.30%
  days edge > 0:          88 / 181

─ Layer B (position) ──────────────────────────────────────────────
  FGY (total):             0.677
  FGY (diffusion only):    0.813
  FGY (jump only):         4.058
  Σ jump_gc / Σ gc_total:  16.7%

─ Layer C (regime) ────────────────────────────────────────────────
  mean cv_realized_il:     0.433  (baseline 0.354)
  mean jump_intensity:     15.6%

─ HAR-RV anti-jump rule ────────────────────────────────────────────
  cumulative PnL Δ:       $  -3,010,017
  high-jump days skipped: 36/92

═════════════

## §8 — Caveats

- **Pool**: USDC/WETH 0.05%, not ETH/USDC 0.3%. Concentration leverage is higher (~17×) on 0.05%; on 0.3% expect 5-8× → IL per dollar TVL lower, fee yield lower too. The `volatility_edge` reading may differ.
- **Sample length**: 180 days. Per spec §10, "Le marché sous-paie la vol aux LPs" sur sample < 6 mois est en limite — verdict to be confirmed on longer windows.
- **HAR-RV** : 60-day rolling out-of-sample (no look-ahead). Coverage starts day 60. For shorter calibration windows, more days but noisier coefficients.
- **BV uses BNS π/2 normalization** so BV/RV → 1 under pure diffusion (otherwise spurious jump attribution).
- **Position synthetic** : L = pool_avg_active_L, range = window-fitted wide. Concentrated narrow ranges would have different IL/fee profile (cf. notebook 11 §6b range-width sensitivity).
- **σ\*_pool** uses V_FR = 2·L·√P denominator (full-range equivalent). Different from σ\*_active which uses concentrated TVL. Both interpretations valid.
- **CV[gamma_cost]** = √RQ / RV — purely subjacent property, range-invariant.
- **Out of scope explicit** (per spec §10): σ*(K) per tick, multi-pool, off-chain hedge.
